In [ ]:
# Run to install shap and joblib
!pip install shap; joblib; lime

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import shap
import joblib
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


c:\Users\HP\anaconda3\envs\machine-learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initialize the data

In [ ]:
# Initialize and access dataset
df = pd.read_csv('data/cleaned_data.csv')

X = df.drop('Attrition', axis=1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Building an RFC model

In [ ]:
# Build an RFC model
rfc = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rfc.fit(X_train, y_train)


In [ ]:
# Tuning the model
parameter_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced"]
}

grid_search = GridSearchCV(
    param_grid=parameter_grid,
    estimator=rfc,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

In [ ]:
# Viewing the best fitting params
print("Best params: ", grid_search.best_params_)
print("Best accuracy score: ", grid_search.best_score_)

# Extracting the best model
tuned_rfc = grid_search.best_estimator_

In [ ]:
# Serializing models
joblib.dump(tuned_rfc, 'rfc_model.joblib')
joblib.dump(X_train.columns.tolist(), 'feature_columns.joblib')

Computing SHAP values (for testing purposes)

In [4]:
# Import serialized model
serialized_rfc = joblib.load('rfc_model.joblib')

In [6]:
# Compute SHAP values
explainer = shap.TreeExplainer(
    serialized_rfc,
    X_train,
    feature_perturbation="interventional"
)

shap_values = explainer.shap_values(X_train)

if isinstance(shap_values, list):
    shap_attrition = shap_values[1]
else:
    shap_attrition = shap_values[:, :, 1]


mean_abs_shap_values = np.abs(shap_attrition).mean(axis=0)

print("X_train shape:", X_train.shape)
print("Number of features:", len(X_train.columns))
print("SHAP values type:", type(shap_values))
print("SHAP values shape:", np.array(shap_values).shape)

shap_importance = (
    pd.DataFrame({
        'feature': X_train.columns,
        'mean_abs_shap': mean_abs_shap_values
    }).sort_values(by='mean_abs_shap', ascending=False)
)

shap.summary_plot(shap_attrition, X_train)

NameError: name 'X_train' is not defined

In [ ]:
top_features = shap_importance["feature"].head(5).tolist()
top_features

In [ ]:
employee_idx = 0

employee_shap = shap_attrition[employee_idx]

employee_importance = pd.DataFrame({
    "feature": X_train.columns,
    "shap_value": employee_shap,
    "abs_shap": np.abs(employee_shap)
}).sort_values(by="feature", ascending=False)

print(employee_importance.head(10))

Computing LIME values (for testing purposes)

In [ ]:
# Lime 
lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns.tolist(),
    class_names=['No Attrition', 'Attrition'],
    mode='classification',
)

lime_exp = lime_explainer.explain_instance(
    data_row=X_test.iloc[1].values,
    predict_fn=serialized_rfc.predict_proba,
    num_features=10,
    top_labels=1
)

lime_exp.as_list(label=lime_exp.available_labels()[0])

In [ ]:
# plot the lime graph
lime_exp.as_pyplot_figure(label=lime_exp.available_labels()[0]).show()

Building functions for getting results

In [ ]:
# Evaluation function
def evaluate_model(csv_path, bin_pred, pred_proba):
    # Load saved artifacts
    model = joblib.load('rfc_model.joblib')
    feature_columns = joblib.load('feature_columns.joblib')

    #Load data
    data = pd.read_csv(csv_path)
    X_new = data[feature_columns]

    data['Predicted_Attrition_Proba'] = pred_proba
    data['Predicted_Attrition'] = bin_pred

    # Lime 
    lime_explainer = LimeTabularExplainer(
        training_data=np.array(X_new),
        feature_names=X_new.columns.tolist(),
        class_names=['No Attrition', 'Attrition'],
        mode='classification',
    )

    lime_exp = lime_explainer.explain_instance(
        data_row=X_new.iloc[0].values,
        predict_fn=tuned_rfc.predict_proba,
        num_features=10,
        top_labels=1
    )

    lime_exp.as_list(label=lime_exp.available_labels()[0])

In [ ]:
def individual_values(csv_path, bin_pred, pred_proba, emp_indx):
    # Load saved artifacts
    model = joblib.load('rfc_model.joblib')
    feature_columns = joblib.load('feature_columns.joblib')

    #Load data
    data = pd.read_csv(csv_path)    
    X_new = data[feature_columns]

    data['Predicted_Attrition_Proba'] = pred_proba
    data['Predicted_Attrition'] = bin_pred

    new_explainer = shap.TreeExplainer(
        tuned_rfc,
        X_train,
        feature_perturbation="interventional"
    )

    new_shap_values = explainer.shap_values(X_new)

    if isinstance(new_shap_values, list):
        new_shap_attrition = new_shap_values[1]
    else:
        new_shap_attrition = new_shap_values[:, :, 1]

    employee_idx = 0
    employee_shap = new_shap_attrition[employee_idx]

    employee_importance = pd.DataFrame({
        "feature": X_new.columns,
        "shap_value": employee_shap,
        "abs_shap": np.abs(employee_shap)
    }).sort_values(by="feature", ascending=False)

    return employee_importance.iloc[emp_indx]
